# Lens Notebook

This notebook shows how to use both `LogitLens` and `TunedLens` with `ModelWithSplitPoints` across several model families.

Origins:
- Logit Lens: nostalgebraist, *Interpreting GPT: the logit lens*
- Related vocabulary-space analysis: Geva et al. (2022)
- Tuned Lens: Belrose et al. (2023)

The tuned lens implementation supports three initialization modes:
- `logit_lens`: identity initialization so tuning starts from the plain logit-lens behavior
- `xavier`: Xavier uniform initialization with zero bias
- `default`: the current `torch.nn.Linear` initialization

The notebook uses tiny checkpoints so it stays light enough for quick experimentation.
Because these checkpoints are lightweight fixtures, they can expose uninitialized heads or meta-tensor loading paths that are not representative of the usual workflow.
For practical use, prefer a fully loaded Hugging Face model or a fully materialized `ModelWithSplitPoints`.
Raw text inputs in the examples are tokenized internally by the lens methods with the wrapped tokenizer.
Some of them are random checkpoints, so semantic quality is not the goal here: the examples are mainly meant to illustrate the API and the decodability metrics.

Metric interpretation:
- `mean_target_probability`: higher is better
- `target_cross_entropy`: lower is better
- `perplexity`: lower is better for causal language models
- `kl_divergence_to_model`: lower is better and differentiable, which makes it useful as a regularization target for linear decodability
- `model_top1_agreement`: agreement with the final model argmax

The raw `explain()` and `lens()` outputs are tensor-first and use `top_indices` / `top_scores`.
Human-readable decoding is handled by the visualization layer. For sequence classification, readable class names are passed explicitly rather than inferred automatically.


In [1]:
from transformers import AutoModelForCausalLM, AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer

from interpreto import LogitLens, ModelWithSplitPoints, TunedLens
from interpreto.visualizations import display_lens_results


def summarize_metrics(metrics, split_point):
    keys = [
        'target_source',
        'nb_evaluated_elements',
        'mean_target_probability',
        'target_cross_entropy',
        'target_accuracy',
        'mean_max_probability',
        'kl_divergence_to_model',
        'model_top1_agreement',
        'perplexity',
    ]
    return {key: metrics[split_point][key] for key in keys if key in metrics[split_point]}


example_sentences = [
    'Interpreto is useful.',
    'Interpreto helps explain models.',
    'Interpreto is helpful',
    'Interpreto is practical',
]
causal_examples = example_sentences[:2]
masked_examples = [example_sentences[0].rstrip('.'), 'Interpreto explains transformers']
classification_examples = example_sentences[2:]
classification_targets = [1, 0]
tuning_texts = [
    example_sentences[0],
    'Interpreto helps explain transformers.',
    'Interpreto makes debugging easier.',
    'Interpreto is practical for analysis.',
]
held_out_examples = [
    'Interpreto helps debug transformers.',
    'Interpreto makes analysis practical.',
]


## Causal Language Model

We start with a small GPT-style model and inspect two prompts at once.


In [2]:
causal_model_name = 'hf-internal-testing/tiny-random-gpt2'
causal_model = AutoModelForCausalLM.from_pretrained(causal_model_name)
causal_tokenizer = AutoTokenizer.from_pretrained(causal_model_name)
if causal_tokenizer.pad_token is None:
    causal_tokenizer.pad_token = causal_tokenizer.eos_token

causal_model_with_split_points = ModelWithSplitPoints(
    causal_model,
    tokenizer=causal_tokenizer,
    split_points='transformer.h.1.mlp',
    batch_size=2,
    device_map='cpu',
)

causal_model_with_split_points.split_points


Loading weights:   0%|          | 0/64 [00:00<?, ?it/s]

['transformer.h.1.mlp']

In [3]:
causal_logit_lens = LogitLens(causal_model_with_split_points, top_k=3)
causal_logit_explanations = causal_logit_lens.lens(causal_examples)


In [4]:
causal_logit_explanations['transformer.h.1.mlp']['top_indices'][0, 0], causal_logit_explanations['transformer.h.1.mlp']['top_scores'][0, 0]


(tensor([962, 929, 551]), tensor([0.0014, 0.0014, 0.0013]))

In [5]:
causal_logit_metrics = causal_logit_lens.metrics(causal_examples)
summarize_metrics(causal_logit_metrics, 'transformer.h.1.mlp')


{'target_source': 'next_token',
 'nb_evaluated_elements': 29,
 'mean_target_probability': 0.0010038625914603472,
 'target_cross_entropy': 6.910143852233887,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0014180837897583842,
 'kl_divergence_to_model': 0.008136783726513386,
 'model_top1_agreement': 0.03448275849223137,
 'perplexity': 1002.3914184570312}

## Masked Language Model

The same `LogitLens` workflow also works on masked-language-model checkpoints.


In [6]:
masked_model_name = 'hf-internal-testing/tiny-random-bert'
masked_model = AutoModelForMaskedLM.from_pretrained(masked_model_name)
masked_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

masked_model_with_split_points = ModelWithSplitPoints(
    masked_model,
    tokenizer=masked_tokenizer,
    split_points='bert.encoder.layer.1.output',
    batch_size=2,
    device_map='cpu',
)

masked_model_with_split_points.split_points


Loading weights:   0%|          | 0/91 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
qa_outputs.bias              | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
qa_outputs.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


/tmp/interpreto-lens-audit-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:852: UserWarning: Module `model.bert.encoder.layer.0.attention` of type `<class 'transformers.models.bert.modeling_bert.BertAttention'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/tmp/interpreto-lens-audit-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:852: UserWarning: Module `model.bert.encoder.layer.0` of type `<class 'transformers.models.bert.modeling_bert.BertLayer'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/tmp/interpreto-lens-audit-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:852: UserWarning: Module `model.bert.encoder.layer.1.attention` of type `<class 'transformers.models.bert.modeling_bert.BertAttention'>` has pr

['bert.encoder.layer.1.output']

In [7]:
masked_logit_lens = LogitLens(masked_model_with_split_points, top_k=4)
masked_logit_explanations = masked_logit_lens.lens(masked_examples)


In [8]:
masked_logit_metrics = masked_logit_lens.metrics(masked_examples)
summarize_metrics(masked_logit_metrics, 'bert.encoder.layer.1.output')


{'target_source': 'token_identity',
 'nb_evaluated_elements': 48,
 'mean_target_probability': 0.0009017205447889864,
 'target_cross_entropy': 7.0173869132995605,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0012729082955047488,
 'kl_divergence_to_model': 2.1701562218368053e-06,
 'model_top1_agreement': 0.9583333134651184}

## Sequence Classification

For classification checkpoints, the same framework exposes intermediate label distributions and classification-oriented scores.
Below, the visualization step is kept separate from `explain()`, and readable class names are provided explicitly.


In [9]:
classification_model = AutoModelForSequenceClassification.from_pretrained(masked_model_name)
classification_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

classification_model_with_split_points = ModelWithSplitPoints(
    classification_model,
    tokenizer=classification_tokenizer,
    split_points='bert.encoder.layer.1.output',
    batch_size=2,
    device_map='cpu',
)

classification_model_with_split_points.split_points


Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
qa_outputs.bias                            | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect iden

['bert.encoder.layer.1.output']

For this BERT sequence-classification checkpoint, `LogitLens` can reuse the model pooler automatically.\n\nIn practice, the classification path can take three forms: a model-specific pooler or transform before a vector head, a sequence-aware head that consumes 3D hidden states directly as in RoBERTa-style classifiers, or a simple vector head that requires an explicit `pooling_strategy`.\nReadable class names are also kept explicit through `label_names={...}`.

In [10]:
classification_logit_lens = LogitLens(classification_model_with_split_points, top_k=2)
classification_model_inputs = classification_tokenizer(
    classification_examples,
    return_tensors='pt',
    padding=True,
    truncation=True,
)
classification_logit_explanations = classification_logit_lens.explain(classification_model_inputs)
classification_label_names = {0: 'negative', 1: 'positive'}
display_lens_results(
    classification_logit_explanations,
    classification_model_inputs,
    tokenizer=classification_tokenizer,
    task=classification_logit_lens.task,
    label_names=classification_label_names,
)


In [11]:
classification_logit_metrics = classification_logit_lens.metrics(
    classification_examples,
    targets=classification_targets,
)
summarize_metrics(classification_logit_metrics, 'bert.encoder.layer.1.output')


{'target_source': 'provided_targets',
 'nb_evaluated_elements': 2,
 'mean_target_probability': 0.4999966025352478,
 'target_cross_entropy': 0.6932074427604675,
 'target_accuracy': 0.5,
 'mean_max_probability': 0.5051708817481995,
 'kl_divergence_to_model': 5.960464477539063e-08,
 'model_top1_agreement': 1.0}

## Tuned Lens On A Small Dataset

The final section fits a `TunedLens` on a tiny text collection for the causal model.
This is only a small demonstration, but it shows how the decodability metrics can be tracked before and after tuning.


In [12]:
supported_modes = ['logit_lens', 'xavier', 'default']
[TunedLens(causal_model_with_split_points, top_k=3, initialization_mode=mode).initialization_mode for mode in supported_modes]


['logit_lens', 'xavier', 'default']

In [13]:
tuned_lens = TunedLens(causal_model_with_split_points, top_k=3, initialization_mode='logit_lens')
pre_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), 'transformer.h.1.mlp')
history = tuned_lens.fit(tuning_texts, epochs=2, batch_size=2)
post_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), 'transformer.h.1.mlp')
history


{'loss': [0.008252160623669624, 0.00788214709609747],
 'split_points': ['transformer.h.1.mlp'],
 'epochs': 2}

In [14]:
{'before': pre_tuning_metrics, 'after': post_tuning_metrics}


{'before': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.0010038625914603472,
  'target_cross_entropy': 6.910143852233887,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.0014180837897583842,
  'kl_divergence_to_model': 0.008136783726513386,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 1002.3914184570312},
 'after': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.0010151939932256937,
  'target_cross_entropy': 6.898152828216553,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.001416579121723771,
  'kl_divergence_to_model': 0.007438413333147764,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 990.4434814453125}}

In [15]:
tuned_lens_explanations = tuned_lens.lens(held_out_examples)


In [16]:
held_out_metrics = tuned_lens.metrics(held_out_examples)
summarize_metrics(held_out_metrics, 'transformer.h.1.mlp')


{'target_source': 'next_token',
 'nb_evaluated_elements': 36,
 'mean_target_probability': 0.001018359325826168,
 'target_cross_entropy': 6.8959479331970215,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.001417913706973195,
 'kl_divergence_to_model': 0.00745380250737071,
 'model_top1_agreement': 0.0,
 'perplexity': 988.2620849609375}